# Multimodal Deepfake Detection Framework
## Phase I: Video Deepfake Detection with Reproducible Evaluation

This notebook implements the complete pipeline:
1. Verify dataset (140K Real and Fake Faces)
2. XceptionNet training for binary deepfake classification
3. Evaluation with standardized metrics (AUC, EER, F1)
4. GradCAM explainability heatmaps

**Hardware**: NVIDIA GPU (tested on A10G / ml.g5.xlarge)

**Dataset**: 140K Real and Fake Faces (Kaggle)

---
## 1. Setup & Environment Verification

In [ ]:
import sys
import os

# Set project root — adjust if needed
PROJECT_ROOT = "/home/ec2-user/SageMaker/multimodal-deepfake-detection"
os.environ["PROJECT_ROOT"] = PROJECT_ROOT
sys.path.insert(0, PROJECT_ROOT)

import torch
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow.")

In [ ]:
# Install dependencies if needed
!pip install -q -r {PROJECT_ROOT}/requirements.txt

---
## 2. Verify Dataset

Confirm the 140K Real and Fake Faces dataset is in place.

If this cell shows 0 images, go back to STEP_BY_STEP_INSTRUCTIONS.md Part 4, Step 4.4.

In [ ]:
from pathlib import Path
from src.config import FACES_DIR

real_dir = FACES_DIR / "real"
fake_dir = FACES_DIR / "fake"

real_count = len(list(real_dir.glob("*"))) if real_dir.exists() else 0
fake_count = len(list(fake_dir.glob("*"))) if fake_dir.exists() else 0

print(f"Dataset location: {FACES_DIR}")
print(f"Real faces: {real_count:,}")
print(f"Fake faces: {fake_count:,}")
print(f"Total: {real_count + fake_count:,}")

if real_count == 0 or fake_count == 0:
    print("\nERROR: Dataset not found!")
    print("Run the download commands from STEP_BY_STEP_INSTRUCTIONS.md Part 4, Step 4.4")
else:
    print(f"\nDataset ready. Ratio (fake/real): {fake_count/real_count:.2f}")

In [ ]:
# Preview some sample images
import matplotlib.pyplot as plt
from PIL import Image
import random

fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# Show 5 real faces
real_samples = random.sample(list(real_dir.glob("*")), min(5, real_count))
for i, img_path in enumerate(real_samples):
    img = Image.open(img_path)
    axes[0, i].imshow(img)
    axes[0, i].set_title("REAL", color="green", fontweight="bold")
    axes[0, i].axis("off")

# Show 5 fake faces
fake_samples = random.sample(list(fake_dir.glob("*")), min(5, fake_count))
for i, img_path in enumerate(fake_samples):
    img = Image.open(img_path)
    axes[1, i].imshow(img)
    axes[1, i].set_title("FAKE", color="red", fontweight="bold")
    axes[1, i].axis("off")

plt.suptitle("Sample Images from Dataset", fontsize=14)
plt.tight_layout()
plt.show()

# Check image dimensions
sample_img = Image.open(real_samples[0])
print(f"\nImage dimensions: {sample_img.size}")
print(f"Image mode: {sample_img.mode}")

---
## 3. Model Training

Fine-tune XceptionNet (pretrained on ImageNet) for binary deepfake classification.

**Expected time: 2-3 hours** on A10G for 20 epochs with early stopping.

You can watch the loss/accuracy print after each epoch. The model auto-saves the best checkpoint.

In [ ]:
from src.train import train

model, history, split_info, test_loader = train()

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt
from pathlib import Path

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['train_acc'], label='Train')
axes[1].plot(epochs, history['val_acc'], label='Val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, history['val_auc'], label='Val AUC', color='green')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('AUC')
axes[2].set_title('Validation AUC')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(Path(PROJECT_ROOT) / 'results' / 'metrics' / 'training_curves.png'), dpi=150)
plt.show()
print("Saved to results/metrics/training_curves.png")

---
## 4. Evaluation

Evaluate on held-out test set with standardized metrics: AUC, EER, F1, precision, recall.

Generates ROC curve and confusion matrix plots.

In [ ]:
from src.evaluate import evaluate

results = evaluate(test_loader)

---
## 5. GradCAM Explainability

Generate heatmaps showing which facial regions the model attends to when making predictions.

These serve as explainability artifacts for human reviewers and forensic reports.

In [ ]:
from src.evaluate import load_best_model
from src.gradcam import generate_heatmaps

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, _ = load_best_model(device)

generate_heatmaps(model, test_loader, num_samples=50, device=device)

---
## 6. Review Results

Display the evaluation plots and sample heatmaps.

In [ ]:
from src.config import HEATMAPS_DIR, METRICS_DIR
from IPython.display import display, Image as IPImage

# Show ROC curve
print("ROC Curve:")
display(IPImage(filename=str(METRICS_DIR / 'roc_curve.png')))

print("\nConfusion Matrix:")
display(IPImage(filename=str(METRICS_DIR / 'confusion_matrix.png')))

In [ ]:
# Show first 6 heatmap samples
heatmap_files = sorted(HEATMAPS_DIR.glob("*.png"))[:6]
print(f"Showing {len(heatmap_files)} sample GradCAM heatmaps:\n")
for hf in heatmap_files:
    display(IPImage(filename=str(hf)))

In [ ]:
# Print final summary
import json

with open(METRICS_DIR / 'evaluation_report.json') as f:
    report = json.load(f)

print("=" * 50)
print("FINAL RESULTS SUMMARY")
print("=" * 50)
print(f"Model:       {report['model_name']}")
print(f"Test AUC:    {report['test_auc']:.4f}")
print(f"Test EER:    {report['test_eer']:.4f}")
print(f"Test F1:     {report['test_f1']:.4f}")
print(f"Precision:   {report['test_precision']:.4f}")
print(f"Recall:      {report['test_recall']:.4f}")
print(f"Accuracy:    {report['test_accuracy']:.4f}")
print(f"Test Samples: {report['num_test_samples']}")
print("=" * 50)
print("\nPipeline complete! Results saved to results/ directory.")
print("Next steps:")
print("  1. Push to GitHub (see STEP_BY_STEP_INSTRUCTIONS.md Part 6)")
print("  2. STOP your SageMaker instance to avoid charges!")